In [0]:
from pyspark.sql import functions as F

# Dataset location
BASE_PATH = "/Volumes/workspace/default/cic_ids2017/raw"

# Retrieve CSV files
files = sorted([
    f for f in dbutils.fs.ls(BASE_PATH)
    if f.name.lower().endswith(".csv")
])

print("Number of CSV files:", len(files))

if len(files) != 8:
    raise ValueError(f"Expected 8 CSV files, found {len(files)}")

# Dataset validation
statistics = []
reference_columns = None

for file in files:

    print(f"\nInspecting: {file.name}")

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .load(file.path)
    )

    columns = [
        c.strip().replace("\ufeff", "")
        for c in df.columns
    ]

    # Count actual records
    row_count = df.count()

    # Identify label column
    label_columns = [
        c for c in columns
        if c.lower() == "label"
    ]

    # Compare schemas
    current_columns = set(columns)

    if reference_columns is None:
        reference_columns = current_columns

    missing = reference_columns - current_columns
    extra = current_columns - reference_columns

    statistics.append((
        file.name,
        round(file.size / (1024 ** 2), 2),
        row_count,
        len(columns),
        ", ".join(label_columns) if label_columns else "NOT FOUND",
        ", ".join(sorted(missing)) or "None",
        ", ".join(sorted(extra)) or "None"
    ))

# Display validation results
schema = """
file STRING,
size_mb DOUBLE,
row_count LONG,
column_count INT,
label_column STRING,
missing_columns STRING,
extra_columns STRING
"""

validation_df = spark.createDataFrame(statistics, schema)

display(validation_df)

In [0]:
total_rows = sum(row[2] for row in statistics)
total_size = sum(row[1] for row in statistics)

print("========== DATASET SUMMARY ==========")
print("Number of CSV files:", len(statistics))
print("Total records:", f"{total_rows:,}")
print("Total size:", round(total_size, 2), "MB")

print(
    "Average records per file:",
    f"{total_rows // len(statistics):,}"
)